# Personal Knowledge Worker (RAG)

This notebook builds a lightweight **Retrieval-Augmented Generation (RAG)** system.

A user can upload a document (such as a resume or notes), which is:

1. Loaded and split into chunks
2. Converted into embeddings using **OpenAI Embeddings**
3. Stored in a **Chroma vector database**

When a question is asked, the system:

- Retrieves the most relevant document chunks
- Sends the retrieved context to an LLM
- Generates an answer grounded in the uploaded knowledge

To avoid unnecessary recomputation, the vector database is tied to a **content fingerprint of the uploaded file**.  
If the file content remains unchanged, the existing vector collection is reused.

In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import gradio as gr



load_dotenv(override=True)

# -----------------------------
# Config
# -----------------------------
UPLOAD_DIR = "uploaded_kb"
DB_NAME = "vector_db"
COLLECTION_PREFIX = "resume_kb"
MODEL_NAME = "gpt-5"

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(DB_NAME, exist_ok=True)

# OpenAI clients
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm_client = ChatOpenAI(temperature=0, model_name=MODEL_NAME)

# Tokenizer for chunk inspection
encoding = tiktoken.get_encoding("cl100k_base")

# Keep currently active vector store in memory for chat calls
ACTIVE_VECTORSTORE = None
ACTIVE_RETRIEVER = None
ACTIVE_COLLECTION = None
ACTIVE_FILEPATH = None



# -----------------------------
# Helpers
# -----------------------------
def file_fingerprint(filepath: str) -> str:
    """
    Stable fingerprint from file CONTENT, not temp upload name.
    No hashlib needed.
    """
    with open(filepath, "rb") as f:
        data = f.read()

    arr = np.frombuffer(data, dtype=np.uint8)
    size = len(arr)

    if size == 0:
        return "empty"

    byte_sum = int(arr.astype(np.uint64).sum())

    # weighted checksum for a better signature than plain sum
    positions = np.arange(1, size + 1, dtype=np.uint64)
    weighted_sum = int((arr.astype(np.uint64) * positions).sum() % np.uint64(10**12 + 39))

    return f"{size}_{byte_sum}_{weighted_sum}"


def build_collection_name(filepath: str) -> str:
    base = os.path.splitext(os.path.basename(filepath))[0].lower().replace(" ", "_")
    fp = file_fingerprint(filepath)
    return f"{COLLECTION_PREFIX}_{base}_{fp}"


def load_documents(filepath: str):
    """
    Uses TextLoader.
    Best for .txt / .md / plain text files.
    """
    loader = TextLoader(filepath, encoding="utf-8", autodetect_encoding=True)
    return loader.load()


def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50,
        separators=["\n\n", "\n", " ", ""]
    )
    return splitter.split_documents(documents)


def count_tokens(text: str) -> int:
    return len(encoding.encode(text))


def build_or_load_vectordb(filepath: str):
    global ACTIVE_VECTORSTORE, ACTIVE_COLLECTION, ACTIVE_FILEPATH, ACTIVE_RETRIEVER

    collection_name = build_collection_name(filepath)

    # Reuse current in-memory instance if same file contents
    if ACTIVE_COLLECTION == collection_name and ACTIVE_VECTORSTORE is not None:
        return ACTIVE_VECTORSTORE, collection_name, "Reusing active in-memory collection."

    # Load or create Chroma collection
    vectorstore = Chroma(
        collection_name=collection_name,
        persist_directory=DB_NAME,
        embedding_function=embeddings,
    )

    # If collection is empty, populate it
    existing_count = vectorstore._collection.count()

    if existing_count == 0:
        docs = load_documents(filepath)
        chunks = split_documents(docs)

        # add lightweight metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata["source"] = os.path.basename(filepath)
            chunk.metadata["chunk_index"] = i
            chunk.metadata["token_count"] = count_tokens(chunk.page_content)

        vectorstore.add_documents(chunks)
        status = f"Created new collection with {len(chunks)} chunks."
    else:
        status = f"Loaded existing collection with {existing_count} chunks."

    ACTIVE_VECTORSTORE = vectorstore
    ACTIVE_COLLECTION = collection_name
    ACTIVE_FILEPATH = filepath
    ACTIVE_RETRIEVER = vectorstore.as_retriever()

    return vectorstore, collection_name, status


def retrieve_context(query: str, k: int = 4):
    if ACTIVE_VECTORSTORE is None:
        return [], "Please upload a text file first."

    docs = ACTIVE_RETRIEVER.invoke(query, k=k)

    if not docs:
        return [], "No relevant context found."

    context = "\n\n".join(
        [
            f"[Chunk {doc.metadata.get('chunk_index', '?')}] {doc.page_content}"
            for doc in docs
        ]
    )

    return context


def answer_question(message, history):
    if ACTIVE_VECTORSTORE is None:
        return "Upload your resume or knowledge file first."

    context = retrieve_context(message, k=4)

    if isinstance(context, str) and context.startswith("Please upload"):
        return context

    system_prompt = """You answer questions using only the retrieved context.
If the answer is not in the context, say you do not see it in the uploaded knowledge base.
Be concise and accurate.
"""

    user_prompt = f"""Retrieved context:
{context}

Question:
{message}
"""
    response = llm_client.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])

    return response.content



In [ ]:
def save_uploaded_file(uploaded_file) -> str:
    """
    Save uploaded Gradio file to UPLOAD_DIR using a stable filename.
    """
    original_name = os.path.basename(uploaded_file)
    destination = os.path.join(UPLOAD_DIR, original_name)

    with open(uploaded_file, "rb") as src, open(destination, "wb") as dst:
        dst.write(src.read())

    return destination


def upload_and_index(file):
    if file is None:
        return "No file uploaded."

    # Gradio gives us a temp filepath when type='filepath'
    saved_path = save_uploaded_file(file)

    ext = os.path.splitext(saved_path)[1].lower()
    if ext not in [".txt", ".md"]:
        return (
            f"Uploaded: {os.path.basename(saved_path)}\n\n"
            "For this zero-install version, please upload a .txt or .md file.\n"
            "Export your resume to plain text and try again."
        )

    _, collection_name, status = build_or_load_vectordb(saved_path)

    return (
        f"Uploaded: {os.path.basename(saved_path)}\n"
        f"Collection: {collection_name}\n"
        f"Status: {status}"
    )


# -----------------------------
# Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## Personal Knowledge Worker")
    gr.Markdown("Upload a text resume or note file, index it into Chroma, then ask questions.")

    with gr.Row():
        upload_btn = gr.UploadButton(label="Upload knowledge file (.txt or .md)", file_types=[".txt", ".md"])
        upload_status = gr.Textbox(label="Status", lines=6)


    upload_btn.upload(
        fn=upload_and_index,
        inputs=upload_btn,
        outputs=upload_status
    )

    chat_ui = gr.ChatInterface(
        fn=answer_question,
        type="messages"
    )

demo.launch()